# flybench in five minutes (toy connectome)

The benchmark, end to end, on the hand-wired 3.6k-neuron toy that ships with the code — no download. Runtime ≈ 2 minutes on Colab. The two real connectomes need a Codex / neuPrint export (see the README) and ~2 GB of RAM; this notebook shows the machinery.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brandoncho369/flybench/blob/master/notebooks/01_toy.ipynb)

In [ ]:
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/brandoncho369/flybench'], check=True)
import flybench, os
os.environ['PYTHONUTF8'] = '1'
print('flybench', flybench.__version__)

## 1. A connectome, a model, a stimulus

Every neuron is the same five-constant leaky integrate-and-fire unit (Shiu et al. 2024). A stimulus is a Poisson drive on a population chosen by annotation; a readout is a population's mean rate.

In [ ]:
from flybench import load_connectome, LIFSimulator, LIFParams, Stimulus
c = load_connectome('toy')
print(c.n, 'neurons,', c.n_edges, 'edges')
sugar = c.select('GRN_sugar'); mn9 = c.select('MN9')
sim = LIFSimulator(c, LIFParams(gain=1.0, seed=0))
res = sim.run(1000, [Stimulus(sugar, rate_hz=100, t_start_ms=200, t_end_ms=700)])
print('MN9 during sugar: %.0f Hz; after: %.0f Hz' % (res.rate_hz(mn9, 200, 700), res.rate_hz(mn9, 800, 1000)))

## 2. The suite

31 tasks, each a YAML file with a citation and thresholds whose `basis` is a published number or a stated convention. The toy is wired so that every circuit-level task can pass; the six that need dynamics a uniform LIF does not have (dose response, adaptation, one giant-fiber spike per loom, the PN transfer function) fail on it too, which is the model, not the wiring. `--controls rewired` scores each task again on shuffled wiring: the difference is the task's specificity.

In [ ]:
from flybench.bench import load_tasks, run_suite, leaderboard
rep = run_suite(c, LIFParams(gain=1.0, seed=0), load_tasks(), seeds=1, controls=['rewired'])
print('core %.2f  hard %.2f  graded %.2f  specificity %+.2f' % (rep['core_score'], rep['hard_score'], rep['graded'], rep['specificity']))
for t in rep['tasks']:
    print('%-32s %s  %.0f%%  rewired %.0f%%' % (t['task'], 'PASS' if t['passed'] else 'fail', 100*t['score'], 100*t['controls']['rewired']['score']))

## 3. Break it

A task is a claim about wiring. Cut the edge the claim rests on and the task should fail — that is the toy's second job.

In [ ]:
import numpy as np, scipy.sparse as sp
from flybench.connectome import Connectome
from flybench.bench import run_task
W = c.W.tolil()
for i in c.select('GRN_bitter'):
    for j in c.select('bitter_ln'): W[i, j] = 0.0        # the bitter receptors no longer reach their local neurons
broken = Connectome(root_ids=c.root_ids, W=sp.csr_matrix(W, dtype=np.float32), positions=c.positions, annotations=c.annotations, name='toy', meta=dict(c.meta))
t = next(t for t in load_tasks() if t['name'] == 'bitter_suppression')
for name, cc in (('intact', c), ('bitter pathway cut', broken)):
    r = run_task(t, cc, LIFParams(gain=1.0, seed=0))
    print(name, 'PASS' if r.passed else 'fail', [round(ch.value, 2) for ch in r.checks])

## 4. Your model

Any object with `run(duration_ms, stimuli) -> SimResult` is a simulator. `flybench verify-adapter module:Class` checks the contract. The real question the benchmark asks of it is in `docs/FINDINGS.md`: which listed behaviours a mechanism buys, and which it costs.

In [ ]:
from flybench.adapter import verify_adapter
from flybench.models.adaptive_lif import AdaptiveLIFSimulator
rep = verify_adapter(AdaptiveLIFSimulator, c)
print(rep.simulator, 'conforms' if rep.ok else [d.name for d in rep.defects])